# Notebook 02: DPO Training & Evaluation

Train DPO on Qwen2.5-3B with preference data, then compare:
- **Base**: Qwen2.5-3B-Instruct (zero-shot)
- **SFT**: Phase 1 fine-tuned model
- **SFT + DPO**: SFT with DPO alignment (this notebook)

Includes ablation studies on beta and data size.

**Requirements**: Colab Pro with L4 GPU (~14GB VRAM)

### Resume Guide

If the DPO model is already trained and saved on Drive, you can skip Part 1:

| Section | Action |
|---------|--------|
| Setup (cells 2-6) | Run |
| Data Repair (cells 8-9) | Run (idempotent) |
| **Part 1 Training (cells 11-13)** | **SKIP if model exists** |
| Part 2 Evaluation (cells 15-21) | Run |
| Part 3 Judge (cells 23-24) | Run |
| Part 4 Safety (cells 26-27) | Run |
| Part 5 Ablation (cells 29-35) | Run (auto-resume) |
| Save Results (cell 37) | Run |

## Setup

In [ ]:
!pip install -q torch transformers peft trl bitsandbytes accelerate datasets openai tqdm matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/smallllm-dpo'
DRIVE_DIR = '/content/drive/MyDrive/smallllm-dpo'

if os.path.exists(DRIVE_DIR):
    !cp -r {DRIVE_DIR} {PROJECT_DIR}
else:
    !git clone https://github.com/XIECHENG6/smallllm-dpo.git {PROJECT_DIR}

os.chdir(PROJECT_DIR)

import sys
sys.path.insert(0, PROJECT_DIR)

### GPU Check & Model Configuration

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# --- Model Configuration (edit once, used by all cells below) ---
# Phase 1 SFT adapter is NOT compatible (trained with custom prompt format,
# but this project uses Qwen chat template via apply_chat_template).
# Set to None to train DPO directly on the base model.
SFT_ADAPTER = None

### Load Preference Dataset

In [ ]:
# Load preference dataset from Notebook 01
from datasets import load_from_disk

train_ds = load_from_disk('data/train_dataset')
test_ds = load_from_disk('data/test_dataset')
print(f'Train: {len(train_ds)}, Test: {len(test_ds)}')
print(f'Columns: {train_ds.column_names}')

# Inspect one sample (conversational format)
sample = train_ds[0]
print(f'\nChosen conversation ({len(sample["chosen"])} messages):')
for msg in sample['chosen']:
    role = msg['role']
    content = msg['content'][:80] + '...' if len(msg['content']) > 80 else msg['content']
    print(f'  [{role}] {content}')

## Data Repair

Remove identical chosen/rejected pairs produced by Notebook 01.
Programmatically regenerate rejected responses using `_create_malformed_value()`.
This cell is idempotent - safe to re-run.

In [ ]:
import json
import random
from datasets import Dataset
from src.data.generator import ScenarioGenerator
from src.data.formatter import load_pairs_jsonl, save_pairs_jsonl, pairs_to_dataset

# --- Step 1: Identify identical pairs ---
pairs = load_pairs_jsonl('data/preference_pairs.jsonl')
identical_idx = [i for i, p in enumerate(pairs) if p['chosen'] == p['rejected']]
print(f'Found {len(identical_idx)} identical pairs to repair')

# --- Step 2: Programmatically fix rejected responses ---
gen = ScenarioGenerator(llm_client=None, seed=42)

repaired = 0
for i in identical_idx:
    pair = pairs[i]
    correct_dict = json.loads(pair['chosen'])
    corrupted = gen._create_malformed_value(correct_dict)
    new_rejected = json.dumps(corrupted, ensure_ascii=False)
    if new_rejected != pair['chosen']:
        pair['rejected'] = new_rejected
        pair['error_type'] = 'malformed_json_value'
        repaired += 1
    else:
        pairs[i] = None  # Mark for removal if corruption failed

pairs = [p for p in pairs if p is not None]
print(f'Repaired: {repaired}, Removed: {len(identical_idx) - repaired}, Final: {len(pairs)}')

# --- Step 3: Verify no identical pairs remain ---
still_identical = sum(1 for p in pairs if p['chosen'] == p['rejected'])
print(f'Remaining identical pairs: {still_identical}')
assert still_identical == 0, 'Data repair incomplete!'

# --- Step 4: Save repaired data and rebuild datasets ---
save_pairs_jsonl(pairs, 'data/preference_pairs.jsonl')
train_ds, test_ds = pairs_to_dataset(pairs, train_ratio=0.9, seed=42)
train_ds.save_to_disk('data/train_dataset')
test_ds.save_to_disk('data/test_dataset')
print(f'\nRepaired dataset: {len(train_ds)} train / {len(test_ds)} test')

# --- Step 5: Show error type distribution after repair ---
from collections import Counter
error_dist = Counter(p['error_type'] for p in pairs)
print('\nError type distribution (repaired):')
for et, count in error_dist.most_common():
    print(f'  {et}: {count} ({count/len(pairs)*100:.1f}%)')

## Part 1: DPO Training

> **SKIP cells 12-13 if the model is already trained and saved to Drive.**
> Re-running will overwrite the saved checkpoint.

### 1a. Train SFT + DPO Model

In [ ]:
from src.training.dpo_train import DPOTrainingArgs, train_dpo

# VRAM pre-check
free_vram = torch.cuda.mem_get_info()[0] / 1024**3
total_vram = torch.cuda.mem_get_info()[1] / 1024**3
print(f'VRAM: {free_vram:.1f} / {total_vram:.1f} GB free')
assert free_vram > 10, f'Only {free_vram:.1f}GB free — need >10GB. Try: Runtime > Change runtime type > L4/A100'

args = DPOTrainingArgs(
    base_model='Qwen/Qwen2.5-3B-Instruct',
    sft_adapter=SFT_ADAPTER,
    output_dir='./dpo_output/sft_dpo',
    beta=0.5,
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    max_length=512,
    max_prompt_length=384,
    lora_rank=16,
    lora_alpha=32,
)

trainer, train_metrics = train_dpo(train_ds, test_ds, args)
print('\nTraining metrics:')
for k, v in train_metrics.items():
    print(f'  {k}: {v}')

### 1b. Save Model to Drive & Release GPU

In [ ]:
import gc

# Save to Drive
!cp -r dpo_output/sft_dpo {DRIVE_DIR}/dpo_output/sft_dpo
print('DPO model saved to Drive')

# Release training resources before evaluation
del trainer
gc.collect()
torch.cuda.empty_cache()
print('GPU memory released')

## Part 2: Evaluation - 3-Way Comparison

Compare Base / SFT / SFT+DPO on the same test set.

### 2a. Load Test Scenarios

In [ ]:
from src.evaluation.metrics import load_model_for_eval, evaluate_model, compare_models
from src.data.formatter import load_pairs_jsonl

# Load test scenarios
all_scenarios = load_pairs_jsonl('data/scenarios.jsonl')
test_scenarios = all_scenarios[-200:]  # Last 200 for evaluation
print(f'Evaluating on {len(test_scenarios)} scenarios')

### 2b. Evaluate Each Model

In [ ]:
# Evaluate Base model
base_model, tokenizer = load_model_for_eval('Qwen/Qwen2.5-3B-Instruct')
base_metrics, base_preds, base_refs = evaluate_model(base_model, tokenizer, test_scenarios)
del base_model
torch.cuda.empty_cache()
print('Base:', base_metrics)

In [ ]:
# Evaluate SFT model
if SFT_ADAPTER:
    sft_model, tokenizer = load_model_for_eval(
        'Qwen/Qwen2.5-3B-Instruct', adapter_path=SFT_ADAPTER
    )
    sft_metrics, sft_preds, sft_refs = evaluate_model(sft_model, tokenizer, test_scenarios)
    del sft_model
    torch.cuda.empty_cache()
    print('SFT:', sft_metrics)
else:
    sft_metrics = None
    sft_preds = None
    print('SFT adapter not set, skipping SFT evaluation')

In [ ]:
# Evaluate DPO model (must replicate SFT merge from training)
dpo_model, tokenizer = load_model_for_eval(
    'Qwen/Qwen2.5-3B-Instruct',
    sft_adapter=SFT_ADAPTER,
    adapter_path='./dpo_output/sft_dpo',
)
dpo_metrics, dpo_preds, dpo_refs = evaluate_model(dpo_model, tokenizer, test_scenarios)
del dpo_model
torch.cuda.empty_cache()
print('DPO:', dpo_metrics)

In [ ]:
# Side-by-side comparison
results = {'Base (zero-shot)': base_metrics, 'SFT + DPO': dpo_metrics}
if sft_metrics:
    results['SFT only'] = sft_metrics
compare_models(results)

### 2c. Visualization

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import numpy as np

metrics_names = ['json_valid_rate', 'name_accuracy', 'arg_names_accuracy', 'arg_values_accuracy', 'exact_match']
labels = ['JSON Valid', 'Name Acc', 'Arg Names', 'Arg Values', 'Exact Match']

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(labels))
width = 0.25

bars = []
model_names = list(results.keys())
colors = ['#4A90D9', '#E8833A', '#50C878']

for i, (name, metrics) in enumerate(results.items()):
    values = [metrics[m] * 100 for m in metrics_names]
    offset = (i - len(results)/2 + 0.5) * width
    bar = ax.bar(x + offset, values, width, label=name, color=colors[i % len(colors)])
    bars.append(bar)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Function Calling: Base vs SFT vs SFT+DPO', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/model_comparison.png', dpi=150)
plt.show()

## Part 3: LLM-as-Judge Head-to-Head

In [ ]:
import os
from google.colab import userdata
try:
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
except:
    import getpass
    os.environ['DEEPSEEK_API_KEY'] = getpass.getpass('DeepSeek API key: ')

from src.utils.llm_client import LLMClient
from src.evaluation.judge_eval import JudgeEvaluator

judge_client = LLMClient()
judge = JudgeEvaluator(judge_client)

# Head-to-head: Base vs DPO
judge_scenarios = test_scenarios[:50]  # Use subset to save API costs
h2h_results = judge.evaluate_head_to_head(
    judge_scenarios,
    base_preds[:50], dpo_preds[:50],
    model_a_name='Base', model_b_name='SFT+DPO',
    swap_order=True,
)
judge.print_summary(h2h_results)

In [ ]:
# If SFT model was evaluated, also compare SFT vs DPO
if sft_preds:
    h2h_sft_dpo = judge.evaluate_head_to_head(
        judge_scenarios,
        sft_preds[:50], dpo_preds[:50],
        model_a_name='SFT', model_b_name='SFT+DPO',
        swap_order=True,
    )
    judge.print_summary(h2h_sft_dpo)

## Part 4: Safety Evaluation

In [ ]:
from src.evaluation.safety import evaluate_safety, print_safety_report

# Evaluate DPO model safety (must replicate SFT merge from training)
dpo_model, tokenizer = load_model_for_eval(
    'Qwen/Qwen2.5-3B-Instruct',
    sft_adapter=SFT_ADAPTER,
    adapter_path='./dpo_output/sft_dpo',
)

safety_results_dpo = evaluate_safety(dpo_model, tokenizer)
print('=== SFT + DPO ===')
print_safety_report(safety_results_dpo)

del dpo_model
torch.cuda.empty_cache()

In [ ]:
# Compare with base model safety
base_model, tokenizer = load_model_for_eval('Qwen/Qwen2.5-3B-Instruct')
safety_results_base = evaluate_safety(base_model, tokenizer)
print('=== Base Model ===')
print_safety_report(safety_results_base)

del base_model
torch.cuda.empty_cache()

print(f"\nSafety improvement: {safety_results_base['pass_rate']:.1%} -> {safety_results_dpo['pass_rate']:.1%}")

## Part 5: Ablation Studies

> Supports **auto-resume**: completed sweep values are skipped automatically.
> Safe to re-run after interruption.

### 5a. DPO Beta Sweep

Compare beta = 0.05, 0.1, 0.5 (KL penalty strength).

In [ ]:
# Ablation 1: DPO beta sweep
from src.training.dpo_train import DPOTrainingArgs, run_dpo_sweep

free_vram = torch.cuda.mem_get_info()[0] / 1024**3
print(f'VRAM before beta sweep: {free_vram:.1f} GB free')
assert free_vram > 10, f'Only {free_vram:.1f}GB free - release models or restart runtime'

beta_args = DPOTrainingArgs(
    base_model='Qwen/Qwen2.5-3B-Instruct',
    output_dir='./dpo_output/ablation_beta',
    learning_rate=5e-6,
    num_train_epochs=1,
)

beta_results = run_dpo_sweep(
    train_ds, test_ds, beta_args,
    sweep_param='beta',
    sweep_values=[0.05, 0.1, 0.5],
)

print('\nBeta sweep results:')
for r in beta_results:
    print(f"  beta={r['value']}: loss={r['metrics'].get('train_loss', 'N/A')}")

### 5b. Data Size Scaling

Train with 200, 500, and full dataset to measure data efficiency.

In [ ]:
# Ablation 2: Data size scaling
import gc
import json as _json
from src.training.dpo_train import DPOTrainingArgs, train_dpo

free_vram = torch.cuda.mem_get_info()[0] / 1024**3
print(f'VRAM before data scaling: {free_vram:.1f} GB free')
assert free_vram > 10, f'Only {free_vram:.1f}GB free - release models or restart runtime'

data_sizes = [200, 500, len(train_ds)]
scaling_results = []

for size in data_sizes:
    out_dir = f'./dpo_output/ablation_data_{size}'
    metrics_file = os.path.join(out_dir, 'train_results.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = _json.load(f)
        scaling_results.append({'size': size, 'metrics': metrics})
        print(f'Skipping data={size} (already completed)')
        continue

    subset = train_ds.select(range(min(size, len(train_ds))))
    scale_args = DPOTrainingArgs(
        base_model='Qwen/Qwen2.5-3B-Instruct',
        output_dir=out_dir,
        learning_rate=5e-6,
        num_train_epochs=1,
    )
    trainer, metrics = train_dpo(subset, test_ds, scale_args)
    scaling_results.append({'size': size, 'metrics': metrics})
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

print('\nData scaling results:')
for r in scaling_results:
    print(f"  {r['size']} pairs: loss={r['metrics'].get('train_loss', 'N/A')}")

### 5c. Evaluate Ablation Models

In [ ]:
# Evaluate each ablation model on test scenarios
ablation_eval = {}

for beta_val in [0.05, 0.1, 0.5]:
    adapter = f'./dpo_output/ablation_beta/beta_{beta_val}'
    if os.path.exists(adapter):
        model, tok = load_model_for_eval('Qwen/Qwen2.5-3B-Instruct', adapter_path=adapter)
        m, _, _ = evaluate_model(model, tok, test_scenarios[:100])
        ablation_eval[f'beta={beta_val}'] = m
        del model; torch.cuda.empty_cache()

for size in data_sizes:
    adapter = f'./dpo_output/ablation_data_{size}'
    if os.path.exists(adapter):
        model, tok = load_model_for_eval('Qwen/Qwen2.5-3B-Instruct', adapter_path=adapter)
        m, _, _ = evaluate_model(model, tok, test_scenarios[:100])
        ablation_eval[f'data={size}'] = m
        del model; torch.cuda.empty_cache()

if ablation_eval:
    compare_models(ablation_eval)

## Part 6: Save All Results

Consolidate all metrics and copy to Google Drive.

In [ ]:
# Save all results
import json

all_results = {
    'main_comparison': results,
    'judge_h2h': h2h_results['summary'] if 'h2h_results' in dir() else {},
    'judge_sft_dpo': h2h_sft_dpo['summary'] if 'h2h_sft_dpo' in dir() else {},
    'safety_base': safety_results_base,
    'safety_dpo': safety_results_dpo,
    'ablation_eval': {k: v for k, v in ablation_eval.items()} if 'ablation_eval' in dir() and ablation_eval else {},
}

os.makedirs('results', exist_ok=True)
with open('results/evaluation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)

# Copy to Drive
!cp -r results {DRIVE_DIR}/
!cp -r dpo_output {DRIVE_DIR}/
print('All results and models saved to Drive')